<a href="https://colab.research.google.com/github/Sanika515/AIML/blob/main/4_Implement_an_LSTM_to_generate_text_based_on_a_corpus_of_text_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Load a text corpus (Shakespeare). To use your own text: text = open('myfile.txt').read()
url = 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
path = keras.utils.get_file('shakespeare.txt', origin=url)
text = open(path, 'rb').read().decode('utf-8')[:150000]      # first 100k characters (fast)
print("Characters:", len(text)); print(text[:250])

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Characters: 150000
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



In [ ]:
# 2. Character vocabulary: every unique character -> a number
chars = sorted(set(text)); vocab = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = np.array(chars)
print("Vocabulary size:", vocab)

# 3. Make training samples: 40 characters in -> the NEXT character out
SEQ, STEP = 40, 2
X, y = [], []
for i in range(0, len(text) - SEQ, STEP):
    X.append([char2idx[c] for c in text[i:i + SEQ]])
    y.append(char2idx[text[i + SEQ]])
X, y = np.array(X), np.array(y)
print("X:", X.shape, "y:", y.shape)

Vocabulary size: 61
X: (33320, 40) y: (33320,)


In [ ]:
# 4. Build the model
model = keras.Sequential([
    layers.Input(shape=(SEQ,)),
    layers.Embedding(vocab, 32),                 # char id -> 32-number vector
    layers.LSTM(256),
    layers.Dense(vocab, activation='softmax')    # probability of each next character
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 32)         │         1,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 61)             │         7,869 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 92,253 (360.36 KB)

 Trainable params: 92,253 (360.36 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 5. Train (loss should fall steadily)
history = model.fit(X, y, batch_size=128, epochs=15)

Epoch 1/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 29s 102ms/step - loss: 3.0883
Epoch 2/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 41s 103ms/step - loss: 2.5000
Epoch 3/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 27s 102ms/step - loss: 2.3238
Epoch 4/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 41s 103ms/step - loss: 2.2184
Epoch 5/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 41s 102ms/step - loss: 2.1401
Epoch 6/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 29s 109ms/step - loss: 2.0811
Epoch 7/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 27s 101ms/step - loss: 2.0291
Epoch 8/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 27s 103ms/step - loss: 1.9832
Epoch 9/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 30s 115ms/step - loss: 1.9340
Epoch 10/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 30s 116ms/step - loss: 1.8938
Epoch 11/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 37s 102ms/step - loss: 1.8575
Epoch 12/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 27s 102ms/step - loss: 1.8265
Epoch 13/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 26s 101ms/step - loss: 1.7940
Epoch 14/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 26s 101ms/step - loss: 1.7653
Epoch 15/15
261

In [ ]:
# 6. Generate new text, one character at a time
def sample(probs, temperature=0.7):
    probs = np.log(np.asarray(probs, dtype='float64') + 1e-9) / temperature
    probs = np.exp(probs); probs /= probs.sum()
    return np.random.choice(len(probs), p=probs)

def generate(seed, length=300, temperature=0.7):
    out = seed.rjust(SEQ)                                   # pad seed to 40 chars
    for _ in range(length):
        ids = np.array([[char2idx.get(c, 0) for c in out[-SEQ:]]])
        probs = model(ids, training=False).numpy()[0]
        out += idx2char[sample(probs, temperature)]
    return out.strip()

print(generate("ROMEO:", 300, 0.5))
print("\n---- higher temperature = more random ----")
print(generate("ROMEO:", 200, 1.2))

ROMEO:
He wond the recleate! mo the ward
Corisiladus leathy that the brouse the prifed
Have be fods the ceromt:

COMINIUS:
The lodse the love: sither maping well thell,
If voult dearither, for it the gows wave here.

MENENIUS:
Hat it counst hook
The stall thair woron them guve homer;
He of knoust o' to th

---- higher temperature = more random ----
ROMEO:
Your rave a doon.

MENENIUS:
Nhands apleatither
Honf deargrus! Wo thale dead-yo! hid jompe
The katn, Gid? thigh ullofine: when faesh preatts conseain of crif
I; wede prighonts--haingerses.

BuRTugs:


In [ ]:
1. What is the problem?
It is the same task as Problem 3. We give the computer Shakespeare's text, it learns the style, and it then writes new text one character at a time. The only difference is that the layer that reads the text is now an LSTM instead of a SimpleRNN.

2. What is the solution?
Why change from RNN to LSTM? A SimpleRNN forgets. Its memory fades as the text gets longer (the vanishing gradient problem). An LSTM (Long Short-Term Memory) was invented to fix exactly this. It has a separate long-term memory and gates that decide what to keep and what to throw away.

Analogy: a SimpleRNN is like a person who only remembers the last few words of a conversation. An LSTM is like a person with a notebook: they decide what to write down, what to cross out, and what to read out loud.

The LSTM has two memories and three gates:

Part	Meaning in simple words
Cell state (c)	The long-term memory, like a notebook running through the whole sequence
Hidden state (h)	The short-term output passed to the next step
Forget gate (f)	Decides what to erase from the notebook
Input gate (i)	Decides what new information to write into the notebook
Output gate (o)	Decides how much of the notebook to show as output

Each gate uses a sigmoid (output 0 to 1), which works like a valve: 0 means block everything, 1 means let everything pass.

LSTM equations:

c(t) = f(t) × c(t−1) + i(t) × c̃(t) (update the long-term memory)
h(t) = o(t) × tanh( c(t) ) (produce the output)

Here c̃(t) is the candidate, the new information, computed with a tanh.

Why this helps: the cell state passes through the steps with very small changes, so important information from early characters isn't lost. This is how the LSTM avoids vanishing gradients.

####CODE##########
The code

Almost everything is the same as Problem 3. The only real change is one line:

python
model = keras.Sequential([
    layers.Input(shape=(SEQ,)),
    layers.Embedding(vocab, 32),
    layers.LSTM(256),                          # <-- was layers.SimpleRNN(256) in Problem 3
    layers.Dense(vocab, activation='softmax')
])

The other steps, in short:

Load the text (first 150,000 characters of Shakespeare).
Build the vocabulary: char2idx (character → number) and idx2char (number → character).
Make samples: 40 characters are the input X, and the next character is the target y, sliding by 2 characters each time.
Compile with Adam and sparse categorical cross-entropy.
Train for 20 epochs, batch size 128.
Generate text: give a seed ("ROMEO:"), predict the next character, pick one using probabilities and temperature, append it, and repeat 300 times.

Temperature in one line: low (0.5) gives safe, readable text, and high (1.0) gives more creative but messier text.

Parameter count (a good viva point):

LSTM(256) = 4 × 256 × (32 + 256 + 1) = 295,936 parameters.
The "4" is because there are four sets of weights: forget gate, input gate, candidate, and output gate.
A SimpleRNN with the same 256 units has only 73,984. So the LSTM has about 4× more parameters.
4. The output you will see
First 250 characters of Shakespeare and the vocabulary size (about 60–65).
Model summary with the LSTM layer at 295,936 parameters.
Training log: the loss falls steadily over 20 epochs (roughly from 2.6 to around 1.0–1.3).
Generated text for ROMEO: at temperature 0.5. It should look like Shakespeare: character names in capitals followed by a colon, line breaks, and mostly real words.
A second text at temperature 1.0, which is more random.

The LSTM text is usually better than the RNN text: more real words, better line structure, and fewer broken words. It still won't be perfect Shakespeare. This is a small character-level model trained for a short time. This LSTM change was tested for errors but not trained to completion here, so check your own output.

#########BLOCK #################
       Input Characters
              |
              v
          Embedding
              |
              v
        +-------------+
        |  LSTM CELL  |
        |             |
        | Forget Gate |
        | Input Gate  |
        | Output Gate |
        |             |
        | Cell State  |
        +-------------+
              |
              v
         Dense Layer
              |
              v
           Softmax
              |
              v
       Next Character


###########QUE#############
6. RNN vs LSTM (very likely viva question)
		SimpleRNN	                LSTM
Memory		One hidden state, short-term	Cell state (long-term) plus hidden state
Gates		None	                        Forget, input, output
Vanishing
gradient        Suffers badly			Largely solved
Long sequences	Poor				Good
Parameters 	73,984				295,936 (about 4×)
Speed		Faster per step			Slower, but runs fast on a GPU
Text quality	Weaker				Better
7. Questions the examiner may ask
What is an LSTM, and why was it invented? It is an RNN with gates and a cell state, invented to solve the vanishing gradient problem so the network can learn long-term dependencies.
Explain the three gates. The forget gate decides what to erase, the input gate decides what new information to store, and the output gate decides what to expose as output.
Why sigmoid for gates and tanh for the candidate? Sigmoid gives 0 to 1, so it acts like a valve. Tanh gives −1 to 1, a good range for the actual values.
Cell state vs hidden state? The cell state is the long-term memory. The hidden state is the output passed to the next layer and next time step.
Why 4 × 256 × (32+256+1) parameters? There are 4 weight sets (3 gates plus the candidate), each with input weights (32), recurrent weights (256) and a bias (1), for 256 units.
What is GRU? A simpler version of the LSTM with 2 gates (reset and update) and no separate cell state. It has fewer parameters and similar performance.
How can the generated text be improved? More data, more epochs, a bigger or stacked LSTM, and lower temperature.